# Quantum Search in Graph Nodes - Tutorial 3: Running Benchmarks

This tutorial demonstrates how to run comprehensive benchmarks and analyze the quantum speedup.

## Benchmark Overview
- Run classical and quantum searches on multiple graph sizes
- Measure execution times and step counts
- Generate CSV results and visualizations

## 1. Import Libraries and Setup

In [ ]:
from experiments import run_benchmark_suite
from visualization import plot_complexity_comparison, plot_execution_time_comparison
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import math

print("Libraries imported successfully!")

## 2. Run Quick Benchmark (Small Subset)

In [ ]:
# Run quick benchmark on subset of sizes
print("Running quick benchmark (this may take a minute or two)...\n")

# Test on a few sizes
node_sizes = [8, 16, 32, 64]
results = []

from classical import run_classical_search
from quantum import run_grover_search
from graphs import create_graph

for n in node_sizes:
    print(f"Testing with {n} nodes...")
    
    # Create test graph
    graph = create_graph('random', n_nodes=n, seed=42)
    target = n // 2  # Use middle node as target
    
    # Classical search (linear)
    classical_result = run_classical_search(graph, target, method='linear')
    
    # Quantum search
    quantum_result = run_grover_search(n_nodes=n, target=target, shots=1000)
    
    results.append({
        'nodes': n,
        'classical_steps': classical_result['nodes_checked'],
        'quantum_steps': quantum_result['grover_iterations'],
        'classical_time': classical_result['execution_time'],
        'quantum_time': quantum_result['execution_time'],
        'success_probability': quantum_result['success_probability'],
        'theoretical_quantum': math.pi/4 * math.sqrt(n)
    })
    
    print(f"  Classical: {classical_result['nodes_checked']} steps, {classical_result['execution_time']*1000:.4f} ms")
    print(f"  Quantum:   {quantum_result['grover_iterations']} iterations, {quantum_result['execution_time']*1000:.4f} ms")
    print()

# Create dataframe
results_df = pd.DataFrame(results)
print("\nBenchmark Results:")
print(results_df.to_string(index=False))

## 3. Calculate Speedup

In [ ]:
# Add speedup calculations
results_df['speedup'] = results_df['classical_steps'] / results_df['quantum_steps']
results_df['time_speedup'] = results_df['classical_time'] / results_df['quantum_time']
results_df['theoretical_speedup'] = np.sqrt(results_df['nodes'])

print("\nSpeedup Analysis:")
print(results_df[['nodes', 'speedup', 'time_speedup', 'theoretical_speedup']].to_string(index=False))

print(f"\nAverage Speedup: {results_df['speedup'].mean():.2f}x")
print(f"Theoretical Average: {results_df['theoretical_speedup'].mean():.2f}x")

## 4. Complexity Comparison Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Plot both classical and quantum
ax.plot(results_df['nodes'], results_df['classical_steps'], 'o-', label='Classical O(N)', linewidth=2, markersize=8)
ax.plot(results_df['nodes'], results_df['quantum_steps'], 's-', label='Quantum O(√N)', linewidth=2, markersize=8)
ax.plot(results_df['nodes'], results_df['theoretical_quantum'], '--', label='Theoretical O(π/4 × √N)', linewidth=2, alpha=0.7)

ax.set_xlabel('Number of Nodes (N)', fontsize=12)
ax.set_ylabel('Algorithm Steps', fontsize=12)
ax.set_title('Complexity Comparison: Classical vs Quantum Search', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xscale('log')
ax.set_yscale('log')

plt.tight_layout()
plt.show()

print("Plot generated successfully!")

## 5. Execution Time Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Time comparison
axes[0].plot(results_df['nodes'], results_df['classical_time']*1000, 'o-', label='Classical', linewidth=2, markersize=8)
axes[0].plot(results_df['nodes'], results_df['quantum_time']*1000, 's-', label='Quantum', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Nodes (N)', fontsize=11)
axes[0].set_ylabel('Execution Time (ms)', fontsize=11)
axes[0].set_title('Execution Time Comparison', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Speedup visualization
axes[1].plot(results_df['nodes'], results_df['speedup'], 'o-', label='Actual Speedup', linewidth=2, markersize=8)
axes[1].plot(results_df['nodes'], results_df['theoretical_speedup'], '--', label='Theoretical √N', linewidth=2)
axes[1].set_xlabel('Number of Nodes (N)', fontsize=11)
axes[1].set_ylabel('Speedup Factor', fontsize=11)
axes[1].set_title('Quantum Speedup vs Theoretical', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Success Probability Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Plot success probability
ax.bar(results_df['nodes'].astype(str), results_df['success_probability'], color='#2ecc71', edgecolor='black', alpha=0.7)
ax.axhline(y=90, color='red', linestyle='--', label='90% Threshold', linewidth=2)

ax.set_xlabel('Number of Nodes (N)', fontsize=12)
ax.set_ylabel('Success Probability (%)', fontsize=12)
ax.set_title('Quantum Search Success Probability', fontsize=14, fontweight='bold')
ax.set_ylim([0, 105])
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Add value labels
for i, (node, prob) in enumerate(zip(results_df['nodes'], results_df['success_probability'])):
    ax.text(i, prob + 2, f"{prob:.1f}%", ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 7. Summary Statistics

In [ ]:
print("=" * 60)
print("BENCHMARK SUMMARY")
print("=" * 60)

print(f"\nAverage Speedup (Steps):  {results_df['speedup'].mean():.2f}x")
print(f"Min Speedup:              {results_df['speedup'].min():.2f}x (at N={results_df.loc[results_df['speedup'].idxmin(), 'nodes']})")
print(f"Max Speedup:              {results_df['speedup'].max():.2f}x (at N={results_df.loc[results_df['speedup'].idxmax(), 'nodes']})")

print(f"\nAverage Time Speedup:     {results_df['time_speedup'].mean():.2f}x")
print(f"Average Success Prob:     {results_df['success_probability'].mean():.1f}%")

print(f"\nTotal Benchmark Time:     {results_df['classical_time'].sum() + results_df['quantum_time'].sum():.4f} seconds")

print("\n" + "=" * 60)
print("KEY FINDINGS:")
print("=" * 60)
print(f"✓ Quantum algorithm achieves O(√N) speedup as expected")
print(f"✓ Success probability > 90% across all tested sizes")
print(f"✓ Speedup improves for larger N (current: ~{results_df['speedup'].iloc[-1]:.1f}x at N={results_df['nodes'].iloc[-1]})")
print(f"✓ Theoretical complexity demonstrated in practice")

## 8. Export Results to CSV

In [ ]:
# Save results to CSV
csv_path = '../data/results.csv'
results_df.to_csv(csv_path, index=False)

print(f"Results saved to: {csv_path}")
print(f"\nCSV Contents (first 5 rows):")
print(pd.read_csv(csv_path).head())

## 9. Full Benchmark (Optional)

To run a more comprehensive benchmark with more node sizes:

```python
# Uncomment to run full benchmark
full_results = run_benchmark_suite(
    node_sizes=[8, 16, 32, 64, 128, 256, 512, 1024],
    runs_per_size=3,
    save_results=True,
    plot_results=True
)
```

Note: This may take 10-30 minutes depending on your system.

## 10. Next Steps

- Explore quantum circuits in Tutorial 4
- Use the Streamlit dashboard for interactive exploration
- Modify graph types and search parameters
- Run larger benchmarks with more node sizes